In [4]:
from pyspark.sql import SparkSession
from delta import *
from pyspark.sql import functions as F

# 1. CONFIGURAR SPARK + HIVE + DELTA
warehouse_location = "hdfs://hdfs-nn:9000/demo/silver/delta_warehouse"

builder = SparkSession \
    .builder \
    .appName("Titles CSV → Delta Lake Table (Final Cleanup)") \
    .master("local[2]") \
    .config("spark.sql.warehouse.dir", warehouse_location) \
    .config("hive.metastore.uris", "thrift://hive-metastore:9083") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.jars.packages", "io.delta:delta-core_2.12:2.4.0") \
    .enableHiveSupport()

spark = configure_spark_with_delta_pip(builder).getOrCreate()

print("SparkSession iniciada com Hive + Delta Lake.")
db_name = "demo"
table_name = "titles_delta"
delta_path = "hdfs://hdfs-nn:9000/demo/silver/titles_delta"
partition_col = "release_year"
hdfs_path = "hdfs://hdfs-nn:9000/demo/bronze/titles.csv"


# 2. LER O FICHEIRO CSV DA BRONZE, CORRIGIR TIPOS E LIMPAR NULOS

titles_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("multiLine", "true") \
    .option("escape", '\"') \
    .csv(hdfs_path)

print(f"Ficheiro CSV lido com sucesso: {hdfs_path}")

# Aplica correção de tipos
titles_df = titles_df \
    .withColumn("seasons", F.col("seasons").cast("int")) \
    .withColumn("imdb_score", F.col("imdb_score").cast("double"))

# --- NOVA ETAPA: ALINHAR LIMPEZA COM PARQUET ---
print("A aplicar limpeza de nulos/vazios (Null -> 'Unknown')...")
substituicao_str = "Unknown"
string_cols = ["id", "title", "type", "description", "age_certification", 
               "genres", "production_countries", "imdb_id"]

# 1. Substituir valores nulos (Nulls)
titles_df = titles_df.fillna(substituicao_str, subset=string_cols)

# 2. Substituir strings vazias ('')
for col_name in string_cols:
    titles_df = titles_df.withColumn(
        col_name,
        F.when(F.col(col_name) == "", substituicao_str).otherwise(F.col(col_name))
    )
print("Limpeza de nulos/vazios concluída nos campos de texto.")
# --------------------------------------------------


# 3. TRANSFORMAÇÕES DE LIMPEZA E ONE-HOT ENCODING

# 3.1. Limpar Coluna 'genres' e 'production_countries

titles_df = titles_df.withColumn(
    "genres",
    F.regexp_replace(F.col("genres"), r"[\[\]']", "")
)

titles_df = titles_df.withColumn(
    "production_countries",
    F.regexp_replace(F.col("production_countries"), r"[\[\]']", "")
)
# 4. DEFINIÇÕES DA TABELA DELTA E SOBRESCRITA

spark.sql(f"CREATE DATABASE IF NOT EXISTS {db_name}")

# REMOVER AS VERSÕES ANTIGAS E REINSCREVER COM O NOVO ESQUEMA E LIMPEZA
spark.sql(f"DROP TABLE IF EXISTS {db_name}.{table_name}")

print(f"\nA gravar tabela Delta '{db_name}.{table_name}' (sobrescrevendo o histórico) em: {delta_path}")

# ESCREVER COMO TABELA DELTA EXTERNA (CORRETO)
titles_df.write \
    .mode("overwrite") \
    .format("delta") \
    .partitionBy(partition_col) \
    .option("path", delta_path) \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{db_name}.{table_name}")

print(f"Tabela Delta '{db_name}.{table_name}' criada com sucesso com a limpeza alinhada!")


# 5. CONSULTA DE TESTE E HISTÓRICO
print("\nConsulta de teste da tabela Delta (Confirmando 'Unknown'):")
spark.sql(f"""
    SELECT 
        title, 
        genres, 
        age_certification, 
        release_year
    FROM {db_name}.{table_name}
    WHERE age_certification = 'Unknown'
    LIMIT 5
""").show(truncate=False)

print("\nHistórico Delta (Deve haver uma nova versão - a última):")
spark.sql(f"DESCRIBE HISTORY {db_name}.{table_name}").show(truncate=False)

SparkSession iniciada com Hive + Delta Lake.
Ficheiro CSV lido com sucesso: hdfs://hdfs-nn:9000/demo/bronze/titles.csv
A aplicar limpeza de nulos/vazios (Null -> 'Unknown')...
Limpeza de nulos/vazios concluída nos campos de texto.

A gravar tabela Delta 'demo.titles_delta' (sobrescrevendo o histórico) em: hdfs://hdfs-nn:9000/demo/silver/titles_delta
Tabela Delta 'demo.titles_delta' criada com sucesso com a limpeza alinhada!

Consulta de teste da tabela Delta (Confirmando 'Unknown'):
+------------------------------+---------------------------------+-----------------+------------+
|title                         |genres                           |age_certification|release_year|
+------------------------------+---------------------------------+-----------------+------------+
|The Platform                  |scifi, thriller, horror, european|Unknown          |2019        |
|Lunana: A Yak in the Classroom|drama, family                    |Unknown          |2019        |
|Fantastic Fungi      

In [5]:
# A sua tabela está em 'demo.titles_delta'

# 1. ESQUEMA COMPLETO DA TABELA (Mantido)
print("\n--- 1. Esquema Completo da Tabela (Novas Colunas de País) ---")
spark.sql("DESCRIBE demo.titles_delta").show(150, truncate=False)

# 2. AMOSTRA DE DADOS TRANSFORMADOS (AGORA COM IMDb SCORE)
# Vê a limpeza dos géneros, as flags de país E a pontuação IMDb.
print("\n--- 2. Amostra de Dados Transformados (Géneros Limpos, Flags de País e IMDb Score) ---")
display(spark.sql("""
    SELECT
        title,
        genres,
        imdb_score,
        release_year,
        production_countries
    FROM demo.titles_delta
    WHERE release_year >= 1990
    LIMIT 10
""").toPandas())

# 3. ANOS DISTINTOS (Coluna de Partição) (Mantido)
print("\n--- 3. Anos Distintos (Coluna de Partição) ---")
spark.sql("SELECT DISTINCT release_year FROM demo.titles_delta ORDER BY release_year").show(20)


--- 1. Esquema Completo da Tabela (Novas Colunas de País) ---
+-----------------------+---------+-------+
|col_name               |data_type|comment|
+-----------------------+---------+-------+
|id                     |string   |null   |
|title                  |string   |null   |
|type                   |string   |null   |
|description            |string   |null   |
|release_year           |int      |null   |
|age_certification      |string   |null   |
|runtime                |int      |null   |
|genres                 |string   |null   |
|production_countries   |string   |null   |
|seasons                |int      |null   |
|imdb_id                |string   |null   |
|imdb_score             |double   |null   |
|imdb_votes             |double   |null   |
|tmdb_popularity        |double   |null   |
|tmdb_score             |double   |null   |
|# Partition Information|         |       |
|# col_name             |data_type|comment|
|release_year           |int      |null   |
+------------

,title,genres,imdb_score,release_year,production_countries
0,30 Rock,"comedy, sport",8.3,2006,US
1,Friday Night Lights,"drama, sport",8.7,2006,US
2,The IT Crowd,"comedy, european",8.5,2006,GB
3,The Departed,"drama, thriller, crime, action",8.5,2006,US
4,DEATH NOTE,"animation, thriller, scifi, drama, fantasy, crime",9.0,2006,JP
5,Casino Royale,"action, thriller, european",8.0,2006,"DE, GB, US, BS, CZ"
6,The Pursuit of Happyness,"drama, european",8.0,2006,US
7,Code Geass: Lelouch of the Rebellion,"scifi, action, animation, drama, war, thriller",8.7,2006,JP
8,The Lake House,"drama, thriller, fantasy, romance, documentation",6.8,2006,US
9,Blood Diamond,"drama, thriller, action",8.0,2006,"DE, US"



--- 3. Anos Distintos (Coluna de Partição) ---
+------------+
|release_year|
+------------+
|        1945|
|        1954|
|        1956|
|        1958|
|        1959|
|        1960|
|        1961|
|        1963|
|        1966|
|        1967|
|        1969|
|        1971|
|        1972|
|        1973|
|        1974|
|        1975|
|        1976|
|        1977|
|        1978|
|        1979|
+------------+
only showing top 20 rows



In [11]:
spark.stop()